# 🎭 Reconhecimento Facial com TensorFlow
**Pipeline:** MTCNN (detecção) → InceptionV3 + Transfer Learning (classificação)

> ⚠️ Ative a GPU: `Ambiente de execução → Alterar tipo → GPU`

In [ ]:
# 1. Verificar GPU
!nvidia-smi
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2. Instalar dependências
!pip install mtcnn opencv-python-headless

In [ ]:
# 3. Montar Google Drive (dataset e resultados)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Clonar o repositório do projeto
!git clone https://github.com/SEU_USUARIO/face-recognition-project.git
%cd face-recognition-project

In [ ]:
# 5. Pré-processar o dataset (detecção e recorte de faces)
# Descompacte seu dataset antes:
# !unzip /content/drive/MyDrive/dataset_raw.zip -d /content/dataset_raw

!python scripts/preprocess.py \
    --input  /content/dataset_raw/train \
    --output /content/dataset/train \
    --size   160 \
    --margin 20

In [ ]:
# 6. Visualizar algumas faces processadas
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image

dataset_dir = Path('/content/dataset/train')
classes = sorted([d.name for d in dataset_dir.iterdir() if d.is_dir()])
print('Classes encontradas:', classes)

fig, axes = plt.subplots(len(classes), 4, figsize=(12, 3 * len(classes)))
for i, cls in enumerate(classes):
    imgs = list((dataset_dir / cls).glob('*.jpg'))[:4]
    for j, img_path in enumerate(imgs):
        ax = axes[i][j] if len(classes) > 1 else axes[j]
        ax.imshow(Image.open(img_path))
        ax.set_title(cls if j == 0 else '')
        ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Treinar o classificador com Transfer Learning
!python scripts/train.py \
    --data_dir /content/dataset/train \
    --model    inceptionv3 \
    --epochs   30 \
    --batch    32 \
    --lr       1e-4 \
    --output   /content/models/face_classifier.h5

In [ ]:
# 8. Visualizar curvas de treinamento
# (TensorBoard)
%load_ext tensorboard
%tensorboard --logdir /content/models/logs

In [ ]:
# 9. Testar reconhecimento em uma imagem
from IPython.display import Image as IPyImage

# Coloque o caminho de uma imagem de teste
TEST_IMAGE = '/content/test.jpg'

!python scripts/recognize.py \
    --model   /content/models/face_classifier.h5 \
    --source  {TEST_IMAGE} \
    --conf    0.35 \
    --output  /content/result.jpg

IPyImage('/content/result.jpg', width=700)

In [ ]:
# 10. Salvar modelo no Google Drive
import shutil
shutil.copy('/content/models/face_classifier.h5',
            '/content/drive/MyDrive/face_classifier.h5')
shutil.copy('/content/models/class_names.json',
            '/content/drive/MyDrive/class_names.json')
print('✅ Modelo salvo no Google Drive!')